### Imports

In [10]:
from pathlib import Path
import pandas as pd

### Daten einlesen

In [ ]:
df_orders = pd.read_csv("C:/Users/Hagen/OneDrive/HKA/WS26-27/DABI1/Projekt/DABI1-Projekt/data/orders.csv", dtype={"order_id": "string", "customer_id": "string"})


## 2.1 Daten vorbereiten

### a)

In [12]:
# Bestellnummern Eindeutigkeit überprüfen

print("Zeilen:", len(df_orders))
print("Eindeutige Bestellnummern:", df_orders["order_id"].nunique())
print("Fehlende Bestellnummern:", df_orders["order_id"].isna().sum())
print("Spalten:", ", ".join(df_orders.columns))
print("Fehlende Kunden IDs:", df_orders["customer_id"].isna().sum())

Zeilen: 99441
Eindeutige Bestellnummern: 99441
Fehlende Bestellnummern: 0
Spalten: order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date
Fehlende Kunden IDs: 0


In [13]:
# Lieferdatum pro Bestellstatus überprüfen

# Anzahl der Bestellungen und fehlende Lieferdaten pro Bestellstatus ermitteln
liefercheck = (
    orders.groupby("order_status", dropna=False)
    .agg(
        bestellungen=("order_id", "size"),
        lieferdatum_fehlt=("order_delivered_customer_date", lambda s: s.isna().sum()),
    )
)

# Anteil der fehlenden Lieferdaten berechnen
liefercheck["anteil_fehlend_pct"] = (
    100 * liefercheck["lieferdatum_fehlt"] / liefercheck["bestellungen"]
).round(2)

liefercheck.sort_values("bestellungen", ascending=False)

,bestellungen,lieferdatum_fehlt,anteil_fehlend_pct
order_status,,,
delivered,96478,8,0.01
shipped,1107,1107,100.0
canceled,625,619,99.04
unavailable,609,609,100.0
invoiced,314,314,100.0
processing,301,301,100.0
created,5,5,100.0
approved,2,2,100.0


In [14]:
# Überprüfung der Liefertermine für Bestellungen mit Status "delivered"

geliefert = orders.loc[orders["order_status"] == "delivered"].copy()

ist_termin = pd.to_datetime(geliefert["order_delivered_customer_date"], errors="raise")
soll_termin = pd.to_datetime(geliefert["order_estimated_delivery_date"], errors="raise")

mit_ist = ist_termin.notna()
mit_beiden = mit_ist & soll_termin.notna()

print("1. Status delivered:", len(geliefert))
print("2. Davon mit tatsächlichem Liefertermin:", mit_ist.sum())
print("3. Davon auch mit zugesagtem Liefertermin:", mit_beiden.sum())

1. Status delivered: 96478
2. Davon mit tatsächlichem Liefertermin: 96470
3. Davon auch mit zugesagtem Liefertermin: 96470
